# Tutorial, part 1: a protein and two disease variants

This notebook is part 1 of the *hxex* tutorial, step by step. The same steps, with more
explanation, are in the user guide: [Tutorial, part 1](https://github.com/tuttlelm/hxex/blob/main/docs/guide/tutorial.md). Part 2, reading
the fits, is the other notebook in this folder; it continues from the results of this one.

**How to use it.** Click a cell and press **Shift + Enter** to run it; the next cell is then
selected. Run the cells in order, from the top. A line starting with `!` runs an `hxex`
command, exactly as you would type it in a terminal. Most cells that change `hxex.toml` run
`hxex set`, which changes one setting and prints what it changed; the cells that add rules of
several lines use the small helpers in `tutorial_helpers.py`, beside this notebook. The guide
says how to make each change in a text editor instead.

Running a cell again is safe: the cells that change `hxex.toml` find the change already made,
and the commands refuse to replace what they wrote before, and say so. To start again from the
beginning, delete every folder whose name starts with `results` first.

**The data** are from the HDX-MS experiments of: Woods CN, Janowska MK, Ulmer LD, Kaur Sidhu J,
Stone NL, James EI, Guttman M, Bush MF, Klevit RE. Activation mechanism of small heat shock
protein HSPB5 revealed by disease-associated mutants. *Proc Natl Acad Sci USA* 122(20),
e2425061122 (2025). https://doi.org/10.1073/pnas.2425061122. Please cite it if you use the data.

HSPB5 (wild type, D109H and R120G) was labelled at room temperature in 85 % D₂O buffer, pH 7.5,
for 4 s, 1 min, 30 min, and 20 h, with undeuterated and fully deuterated controls. The spectra,
exported from HDExaminer, are in the `SpecExport` folder next to this notebook. The folders also
hold two quench-flow time points (0.1 s and 0.8 s) that are not part of the paper; step 4 leaves
them out.

Run this cell first: it loads *hxex* and the helpers.

In [ ]:
%matplotlib inline
from IPython.display import Image
from tutorial_helpers import add_before

import hxex
from hxex.report import format_table, plot_controls_overview, series_table

## Step 3: write the project file

`hxex init` writes `hxex.toml`, the project file, with every setting at its default value. You
can open it from Jupyter's file list to read it. Its `data_dir` is already `"SpecExport"`, the
folder next to it.

In [ ]:
!hxex init --format specexport

## Step 4: tell hxex about the experiment

A name for the project, two rules that leave out the quench-flow time points (the spectra are
kept, but not fitted), and the back-exchange correction with the labelling buffer's 85 % D₂O.

In [ ]:
add_before("[output]", """
[[input.exclude]]
time = "0.1 s"
note = "quench-flow time point (not in the paper)"

[[input.exclude]]
time = "0.8 s"
note = "quench-flow time point (not in the paper)"
""")
!hxex set hxex.toml project hspb5
!hxex set hxex.toml run.controls.d2o_fraction 0.85

## Step 5: ask for error bars

The bootstrap refits every spectrum many times to give each value a standard error, here with
one and with two populations for every spectrum (`ks = [1, 2]`), which the comparisons need.

In [ ]:
!hxex set hxex.toml run.bootstrap.n_boot 20
!hxex set hxex.toml run.bootstrap.ks "[1, 2]"

## Step 6: check

`hxex check` reads the settings and looks through the folders without analysing anything. It
ends with `ok`. The two warnings at the top are expected: HDExaminer named the variants'
peptide folders from the WT sequence, and *hxex* uses the variants' own FASTA files.

In [ ]:
!hxex check hxex.toml

## Step 7: run

The analysis itself: the spectra are read, the quench-flow ones left out, the isotope peaks
picked, every spectrum fitted with one, two, and three populations, and then bootstrapped. The
bootstrap is the slow part: 20 to 30 minutes on a laptop, about 7 with a GPU. It writes the
bundle (`results`) and the tables (`results_tables`).

In [ ]:
!hxex run hxex.toml

## Step 8: the tables

The tables are CSV files in the `results_tables` folder; open them from Jupyter's file list,
or in a spreadsheet program. `series.csv` has a row for each peptide and charge in each sample:

In [ ]:
results = hxex.load_results("hxex.toml")
print(format_table(series_table(results), limit=10))

## Step 9: the figures

Two peptides of WT, at charge 2: residues 87-96 (one population) and 42-54 (two populations at
the early times). In `series.csv` their `data_id` is 140 and 56. The `[figures]` part of the
project file says which to draw as files, and on which time scale:

In [ ]:
!hxex set hxex.toml figures.series "[140, 56]"
!hxex set hxex.toml figures.scale log

In [ ]:
!hxex report hxex.toml --figures

In the notebook you can also draw any peptide straight away, by sample, residues, and charge.
One population at every time:

In [ ]:
results = hxex.load_results("hxex.toml")  # again, with the new [figures] settings
hxex.show_series(results, "WT", "87-96", 2, kinds=["spectra"])

Two populations at 4 s and 1 min, merging into one by 20 h; the controls (UN, FD) have one each:

In [ ]:
hxex.show_series(results, "WT", "42-54", 2)

**Try it:** change the sample (`"WT"`, `"D109H"`, or `"R120G"`), the residues, or the charge,
and run the cell again. `results.peptides("WT")` lists the peptides of WT. Add `k=1` or `k=2`
to draw every spectrum with that number of populations, or `format="svg"` for figures that
stay sharp when you zoom in (slower for a large spectra page).

In [ ]:
hxex.show_series(results, "R120G", "42-54", 2, kinds=["uptake"])

The controls page of the overview: the retention of every fully deuterated control (the share
of the label it still held when measured; about 0.5 here, against 0.85 in the buffer), and how
each series was corrected.

In [ ]:
plot_controls_overview(results)

## Step 10: compare the variants with WT

`reference = "WT"` compares D109H and R120G with WT; `format = "png"` writes the figures as
pictures.

In [ ]:
!hxex set hxex.toml analysis.reference WT
!hxex set hxex.toml analysis.format png

In [ ]:
!hxex analyse hxex.toml

R120G takes up more deuterium than WT over the first 80 residues, most at the early times:

In [ ]:
Image("results_analysis_WT/butterfly_R120G.png")

The same difference at 1 min, peptide by peptide (the panel below it is for part 2):

In [ ]:
Image("results_analysis_WT/woods_R120G_1min.png")

Every comparison residue by residue: red more deuterium than WT, blue less, grey not significant:

In [ ]:
Image("results_analysis_WT/residue_significance.png")

The tables of the comparison are in `results_analysis_WT`: `compare_peptides.csv`,
`compare_residues.csv`, `species_uptake.csv`, and others. The guide's
[Outputs](https://github.com/tuttlelm/hxex/blob/main/docs/guide/outputs.md) page describes them.

**Next:** part 2, `hxex_b5_tutorial_2_fits.ipynb`, continues from these results.